# Explore the single-species eco-evolutionary models

Change the parameters, run a single-species simulation, and inspect how its
adaptive fitness landscape changes over time. The calculations use the same
ODEs and fitness functions as the paper.

- **Model 1:** higher trait values reduce the intrinsic growth rate.
- **Model 2:** higher trait values reduce carrying capacity.
- Both can use uniform or trait-dependent competition, with evolution on or off.

**Start here:** use the `SEG` environment, then **Run → Run All Cells**. This runs
one default experiment and an optional fixed-trait comparison. 

The blue resident marker is a **circle while living** and an **× of the same
colour after extinction**. The legend updates its abundance and trait at the
selected time. This notebook keeps one resident species per experiment; the
comparison is an independent experiment, not a second interacting species.

For definitions and model combinations see [the model guide](../docs/README.md)

In [ ]:
import importlib
import os
import sys
from pathlib import Path

# Work whether Jupyter starts in the repository root or in notebooks/.
ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "src/models/eco_evo_model.py").is_file()),
    None,
)
if ROOT is None:
    raise RuntimeError("Start Jupyter from this checkout or its notebooks directory.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# The numerical core reads some defaults on import. Each experiment below
# supplies its own model and solver arguments explicitly.
os.environ["SEG_PRESET"] = "model1_f5_BE_evo"

from IPython.display import display

import src.workflows.single_species_exploration as exploration_workflow
import src.workflows.single_species_leader_exploration as leader_workflow
import src.plotting.single_species_exploration as exploration_plotting
import src.plotting.single_species_controls as exploration_controls

# Jupyter caches imported modules across cell reruns. Refresh the notebook
# helpers so edits on disk (including changed function arguments) take effect.
importlib.invalidate_caches()
importlib.reload(exploration_workflow)
importlib.reload(leader_workflow)
importlib.reload(exploration_plotting)
importlib.reload(exploration_controls)
settings_for_scenario = exploration_workflow.settings_for_scenario
landscape_time_slider = exploration_plotting.landscape_time_slider

population_addition_controls = exploration_controls.population_addition_controls
single_species_experiment_controls = exploration_controls.single_species_experiment_controls
leader_analysis_controls = exploration_controls.leader_analysis_controls

# Shared state for the backend controls; later runs invalidate dependent results.
additional_population_stages = {}
leader_panels = {}

print(f"Repository: {ROOT.name}")

## What is being simulated?

The resident has abundance $x(t)$ and trait $u(t)$. A mutant trait $v$ is used to
probe the resident's current fitness landscape:

$$\frac{dx}{dt}=x\,G(u;x,u,E),\qquad
\frac{du}{dt}=\sigma\left.\frac{\partial G(v;x,u,E)}{\partial v}\right|_{v=u}.$$

With evolution disabled, $u$ stays fixed while $x$ changes. The trait-dependent energetic compensation
threshold is $E_c(v)=\alpha/(\beta+v)$; increasing the trait lowers that threshold,
but incurs the demographic cost selected by the model variant.

| Control | Meaning |
|---|---|
| `E` | Effective resource support, held constant during an experiment |
| `r_max`, `K`, `d` | Growth scale, carrying-capacity scale, and mortality |
| `g` | Strength of the trait cost |
| `alpha`, `beta` | Trait-dependent energetic compensation threshold parameters |
| `sigma` | Rate of trait evolution |
| Trait-dependent competition | Enable the competition kernel |
| `beta_bully`, `sigma_a_squared` | Shift and width of the competition kernel |
| Initial `x`, initial `u` | Starting abundance and trait |

Even with one resident, trait-dependent competition can change the selection
gradient. Unchecking trait-dependent competition removes its trait dependence entirely.

The landscape is **instantaneous**: it holds the sampled resident environment
fixed and evaluates mutant fitness across $v$. The resident marker belongs at
$(u(t),G(u(t)))$ and need not lie on the zero-fitness line during a transient.

## Choose a starting example

The examples copy biological parameters from `model1_f5_AD_evo` (Model 1 - only density dependence), `model1_f5_BE_evo` (Model 1 - density and frequency dependence), and `model3_f2_C` (Model 2 - density and frequency dependence). Effective resource support and
simulation duration are exploration settings, independent of the publication
figure recipes. 

You can edit this cell before building the controls, or use the controls below. 

In [ ]:
STARTING_SCENARIO = "model1_density_and_frequency_dependence"
PARAMETER_OVERRIDES = {
    # "E": 0.6,
    # "g": 0.15,
    # "duration": 3000.0,
}
initial_settings = settings_for_scenario(STARTING_SCENARIO, **PARAMETER_OVERRIDES)

## Experiment controls and results

Open the sections to change demographic parameters, competition, and plot
sampling. Press **Run simulation** after making changes.

- The optional dashed curves show the *same* experiment with evolution disabled.
- **Duration** changes the simulated time. **time samples** changes slider
  resolution. Simulations end at the requested duration and may not have stabilised/reached equilibrium.
- Fitness axes stay fixed as you move the time slider. 
- Zooming into the graphs is possible by hovering over with the mouse pointer, left clicking, holding to draw a box to zoom into. 
- Extinction means $x\leq$ the selected threshold (default $10^{-4}$). An extinct resident contributes zero abundance to the
  landscape, and its × shows the retained trait. Fitness at that × describes
  the corresponding trait in the remaining environment.

In [ ]:
population_panel = single_species_experiment_controls(
    initial_settings, STARTING_SCENARIO,
    stages=additional_population_stages, leader_panels=leader_panels,
)

# Live controls and results remain available for interactive inspection.
controls = population_panel["controls"]
experiment = population_panel["experiment"]
run_experiment = population_panel["run"]
load_example = population_panel["load_example"]
scenario_control = population_panel["scenario"]
output_area = population_panel["output"]
compare_control = population_panel["compare"]
full_range = population_panel["full_range"]
y_min, y_max = population_panel["y_min"], population_panel["y_max"]
style, field_layout = population_panel["style"], population_panel["layout"]

display(population_panel["widget"])
run_experiment()


With the default settings for 'Model 1 - density and frequency dependence', you will notice that there is an invasion window - a region of trait space that has positive fitness. But the resident species is sitting on a local peak and is separated from this region by a valley (zoom in on adaptive landscape around the resident). This is not seen for 'Model 1 - only density dependence' where the species is at the global fitness peak. 
In 'Model 2 - density and frequency dependence', the cost of the trait is in the carrying capacity instead of the growth rate. The invasion windows continuously move and the species keeps climbing the fitness gradient to the extent where the cost outweighs the benefit and reduces the carrying capacity so much that it leads to extinction. 

## Leader payoffs and best responses

This section uses the **last successful population experiment** above. Press
**Run leader analysis** to sweep effective resource support using the selected
model, biological parameters, and initial condition. 

The species' best response simulation runs the eco-evolutionary dynamics - it finds the trait value that maximises the fitness for every given $E$. 

The leader's best response simulation runs only the ecological dynamics for a given, fixed trait $u$, and finds the $E$ that maximises the payoff. It does this for every value of $u$ in the given range and finds a corresponding $E$ that maximises payoff.

At the intersection of these two curves, if it exists, is the Nash equilibrium - the $(E,u)$ at which both the leader and the species maximise their payoff and fitness respectively. 

The leader's objective is:

$$Q(E,x,u)=L(1-E)+\frac{p}{\delta}\left(1-e^{-\alpha_L x}\right)+\frac{c_1}{u},$$

where the inverse-trait term applies only to a surviving resident and uses the
existing numerical denominator floor at zero trait. `L` weights the direct
resource-support trade-off; `p`, `delta`, and `alpha_leader` determine the
population benefit; `c1` weights the inverse-trait benefit.

The evo-aware leader anticipates how the trait changes with its decision. The eco-aware
comparison holds the trait fixed while population size re-equilibrates. Both
are evaluated using the same coefficients, including `c1`. Set `c1=0` for both
if you want an objective without a direct trait term. 

### Which trait value to use for the eco-aware leader's payoff?

The eco-aware leader needs to substitute a value for $u$ in their objective.
This value is not an arbitrary trait. For each detected Nash point
$(E_N,u_N,x_N)$, the dashed eco-aware curve uses **that point's trait** $u_N$.
If several Nash points are found, each supplies a separate curve. If none is
found, **no Nash-based fixed-trait curve is drawn**: there is no canonical Nash
trait to substitute, and choosing an initial trait or a convenient number
would change the comparison without a game-theoretic justification.

Let $u^*(E)$ denote the species response and $x^*(E,u)$ the ecological equilibrium.
The plotted payoffs are

$$Q_{\mathrm{evo}}(E)=Q(E,x^*(E,u^*(E)),u^*(E)),$$
$$Q_{\mathrm{eco},N}(E)=Q(E,x^*(E,u_N),u_N).$$

At a Nash equilibrium, $u^*(E_N)=u_N$ and the leader's fixed-trait best response
is $E_N$. Therefore

$$Q_{\mathrm{evo}}(E_N)=Q_{\mathrm{eco},N}(E_N)=Q_N.$$

The two curves meet at the Nash payoff, and $E_N$ maximizes the corresponding
fixed-trait curve. 

### What the figure shows

1. **Payoff versus effective resource support:** the evo-aware curve, the
   Nash-based eco-aware curves when available, and candidate solution markers.
2. **Best responses:** the species trait reached at each $E$, and the leader's
   optimal $E$ for each fixed trait.
3. **Population $x$ versus effective resource support $E$:** the ecological
   population along the evolving response, with Nash-fixed-trait populations
   when available and markers at candidate solutions.

### Why a Nash point may not be found

For some parameterizations, **the best-response curves do not intersect**, so
no living pure-strategy Nash point exists on the examined response branch and
support interval. In particular, the leader's global best response can jump
between survival and extinction policies. 

Failure to find a point on a finite sweep can also be because of a coarse grid,
a short simulation duration, a restricted support interval, or an
unexplored initial-condition branch. Choose a finer grid, extend the simulation times and check that the simulations reach equilibrium (stationarity)
before treating a numerical absence as evidence of nonexistence. The
Stackelberg point can still be identified when no Nash point
is found - because it is the point on the species' best response curve that results in the highest payoff.

The table reports stationarity and residual checks for the candidates.

### Leader's best response in a multi-species scenario
The leader's best response in the multi-species scenarios becomes challenging to visualise and can also require simulations starting from a range of initial conditions to fully characterize. Refer to Figure 4 and 5 in the paper for a more detailed analysis of the leader's payoffs in a 4 species case. 

Below we show how more species can be added and an example of a 3-species ESS. 

In [ ]:
leader_panel = leader_analysis_controls(initial_settings, experiment=experiment)
leader_panels["single_species"] = leader_panel
leader_experiment = leader_panel["experiment"]
run_leader_analysis = leader_panel["run"]

display(leader_panel["widget"])


## Add a second species

For the preset 'Model 1 - density and frequency dependence' above, there was an invasion window once the single species simulation reached eco-evolutionary equilibrium implying an undersaturated community. We can introduce a new species at a small population size close to the resident species, say at $u_2 = 10$. Run this after the single-species experiment above. Species 1 starts from the exact final population and trait of that run. The plots show both population and trait trajectories and the resulting fitness landscape.


In [ ]:
# All population introductions use the imported control builder.
if "additional_population_stages" not in globals():
    additional_population_stages = {}


second_population_default = 10.0   # Species 2 starting abundance
second_trait_default = 10.0       # Species 2 starting trait
second_population_stage = population_addition_controls(
    2, second_population_default, second_trait_default,
    stages=additional_population_stages,
    get_previous_result=lambda: experiment.get("result"),
    get_fitness_limits=population_panel["get_fitness_limits"],
    style=style, layout=field_layout,
)
display(second_population_stage["widget"])

# Retain existing names for inspecting or rerunning the second-population stage.
second_population_control = second_population_stage["population"]
second_trait_control = second_population_stage["trait"]
two_species_experiment = second_population_stage["experiment"]
two_species_output = second_population_stage["output"]
two_species_button = second_population_stage["button"]
run_two_species_experiment = second_population_stage["run"]


## Add a third population

Run the two-species simulation first. Both existing populations start at their exact final populations and traits. We then introduce a new species at $u_3 = 6.1$.


In [ ]:
# Edit these defaults before rerunning this cell, or use the displayed controls.
third_population_default = 10.0
third_trait_default = 6.1
third_population_stage = population_addition_controls(
    3, third_population_default, third_trait_default,
    stages=additional_population_stages,
    get_previous_result=lambda: additional_population_stages.get(2, {}).get("experiment", {}).get("result"),
    get_fitness_limits=population_panel["get_fitness_limits"],
    style=style, layout=field_layout,
)
display(third_population_stage["widget"])


For the defualt settings of 'Model 1 - density and frequency dependence', the system now has reached saturation. There are no more invasion windows and each species sits at a peak of the fitness landscape. This equilibrium is an Evolutionarily Stable State (ESS). 

The leader's best response in the multi-species scenarios becomes challenging to visualise. Refer to Figure 4 and 5 in the paper for a more detailed analysis of the leader's payoffs in these cases.  

## Add a fourth population

With the 'Model 1 - density and frequency dependence' preset, adding a 4th population anywhere still only results in at most 3 species with unique trait values that survive. The 4th species either converges to one of these 3 unique traits or goes extinct. 

However, for other parameterizations, the system may support more species. 

In [ ]:
# Edit these defaults before rerunning this cell, or use the displayed controls.
fourth_population_default = 10.0
fourth_trait_default = 3.0
fourth_population_stage = population_addition_controls(
    4, fourth_population_default, fourth_trait_default,
    stages=additional_population_stages,
    get_previous_result=lambda: additional_population_stages.get(3, {}).get("experiment", {}).get("result"),
    get_fitness_limits=population_panel["get_fitness_limits"],
    style=style, layout=field_layout,
)
display(fourth_population_stage["widget"])
